In [1]:
import pandas as pd
import numpy as np

# 1. Load the merged data (before outlier removal)
df_original = pd.read_csv('production_dataset.csv')

# 2. Perform Outlier Removal (IQR Method)
numeric_cols = df_original.select_dtypes(include=[np.number]).columns
Q1 = df_original[numeric_cols].quantile(0.25)
Q3 = df_original[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

# Define the cleaned dataframe
df_clean = df_original[~((df_original[numeric_cols] < (Q1 - 1.5 * IQR)) | (df_original[numeric_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]

# 3. Get lists of crops
crops_before = df_original['label'].unique()
crops_after = df_clean['label'].unique()

# 4. Find common crops and missing crops
common_crops = set(crops_before).intersection(set(crops_after))
missing_crops = set(crops_before) - set(crops_after)

# 5. Print the Report
print("--- CROP RETENTION REPORT ---")
print(f"Total crop types started with: {len(crops_before)}")
print(f"Total crop types remaining:   {len(crops_after)}")
print(f"\n✅ Number of Common Crops: {len(common_crops)}")

if len(missing_crops) > 0:
    print(f"❌ Crops lost due to outliers: {missing_crops}")
else:
    print("✨ Great news! All crop types were preserved.")

# 6. Show the counts per crop in the cleaned data
print("\n--- Rows per Crop (Clean Data) ---")
print(df_clean['label'].value_counts())

--- CROP RETENTION REPORT ---
Total crop types started with: 18
Total crop types remaining:   15

✅ Number of Common Crops: 15
❌ Crops lost due to outliers: {'Rice', 'Grapes', 'Apples'}

--- Rows per Crop (Clean Data) ---
label
Beans, dry                         24
Cantaloupes and other melons       24
Chick peas, dry                    24
Coffee, green                      24
Coconuts, in shell                 24
Jute, raw or retted                24
Lentils, dry                       24
Seed cotton, unginned              24
Mangoes, guavas and mangosteens    24
Oranges                            24
Papayas                            24
Watermelons                        24
Pigeon peas, dry                   24
Maize (corn)                       21
Bananas                            15
Name: count, dtype: int64


In [2]:
# Save the final 15-crop cleaned data
df_clean.to_csv('final_model_ready_data.csv', index=False)
print(f"Final dataset saved with {len(df_clean)} rows and 15 crops.")

Final dataset saved with 348 rows and 15 crops.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# 1. Load your final 15-crop cleaned data
df = pd.read_csv('cleaned_production_data.csv')

# 2. Separate Features (X) and Target (y)
# We drop 'Element' (it's just text) and 'Value' (what we want to predict)
X = df.drop(['Element', 'Value'], axis=1)

# Convert Crop Labels into numbers (One-Hot Encoding)
X = pd.get_dummies(X, columns=['label'])

y = df['Value']

# 3. Train-Test Split (80% Training, 20% Testing)
# random_state=42 ensures that if you run this again, you get the same split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Initialize the Models
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
}

# 5. Loop to Train and Evaluate each model
print(f"{'Model':<20} | {'R2 Score':<10} | {'MAE (Tonnes)':<15}")
print("-" * 50)

for name, model in models.items():
    # Training the model
    model.fit(X_train, y_train)
    
    # Making predictions on the "Exam" (Test set)
    predictions = model.predict(X_test)
    
    # Calculating Accuracy (R2) and Error (MAE)
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    
    print(f"{name:<20} | {r2:<10.4f} | {mae:<15.2f}")